In [2]:
from dotenv import load_dotenv

load_dotenv()


True

In [3]:
from openai import OpenAI

openai_client = OpenAI()

In [5]:
from pyexpat import model


def llm(prompt):
    response = openai_client.responses.create(
        model = 'gpt-5.4-mini',
        input = prompt
    )
    
    return response.output_text

In [54]:
question = 'I just discovered the course. Can I join now?'

In [8]:
context = """

Do you know my son name?
yes I know its Ikshvak ram.such a cute boy

Did he eat the food well?
No he didn't the food like other kids..

is he cute?
offcourse damn cute...
"""

In [22]:
prompt = f"""

Hey you are an assistant.your task in answer the question as per context..

if you don't know the answer simple say

sorry I don't know

Question:
{question}

Context:
{context}

"""

In [23]:

output = llm(prompt)
output

'No, he didn’t eat the food well.'

In [25]:
def rag(question):
    search_result = search(question)
    user_prompt = prompt(question, search_result)
    return llm(user_prompt)

In [27]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [29]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [31]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [2]:
from minsearch import Index

index = Index(
    text_fields=['question','answer','section'],
    keyword_fields=['course']
)
index.fit(documents)

NameError: name 'documents' is not defined

In [1]:
search_results = index.search(
    question,
    filter_dict={'course':'llm-zoomcamp'},
    num_results=5
)

search_results

NameError: name 'index' is not defined

In [63]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [62]:
def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc['section'])
        lines.append("Q: " + doc['question'])
        lines.append("A: " + doc['answer'])
        lines.append("")
    
    return '\n'.join(lines).strip()


In [48]:
def search(question, course = 'llm-zoomcamp'):
    boost_dict = {'question':2.0, 'answer':1.0, 'section':0.5}
    filter_dict={'course':course}

    return index.search(
        question,
        boost_dict = boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )
    

In [64]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context = context
    )

    return prompt.strip()

In [66]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [67]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [68]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join now. If you want a certificate, though, you need to submit your project while submissions are still open.
